# Sprint 1 · Listas, tuplas, sets y diccionarios

Soluciones a los ejercicios propuestos de las secciones 1 (listas), 2 (tuplas), 3 (sets) y 4 (diccionarios).
Cada celda es independiente: define sus propios datos y se puede ejecutar por separado.

---
# Sección 1. Listas

## Ejercicio 1.1 · Lecturas de un sensor fuera de rango
Separar las lecturas válidas (de -40 a 85 ºC, ambos incluidos) de las descartadas y calcular el porcentaje descartado, la media de las válidas y cuántas veces aparece el código de fallo `-999.0`.

In [1]:
lecturas = [21.5, 22.1, -999.0, 23.4, 150.2, 22.8, -999.0, 24.0, 21.9, 87.3]

validas = [x for x in lecturas if -40 <= x <= 85]
descartadas = [x for x in lecturas if not (-40 <= x <= 85)]

porcentaje = len(descartadas) / len(lecturas) * 100
media = sum(validas) / len(validas)
fallos = lecturas.count(-999.0)

print("Válidas:", validas)
print("Descartadas:", descartadas)
print(f"Porcentaje descartado: {porcentaje:.1f} %")
print(f"Media de las válidas: {media:.2f}")
print("Códigos -999.0 (sensor caído):", fallos)

Válidas: [21.5, 22.1, 23.4, 22.8, 24.0, 21.9]
Descartadas: [-999.0, 150.2, -999.0, 87.3]
Porcentaje descartado: 40.0 %
Media de las válidas: 22.62
Códigos -999.0 (sensor caído): 2


## Ejercicio 1.2 · Quitar duplicados conservando el orden de llegada
Solo con listas: se añade el id a `unicos` únicamente si todavía no está.

In [2]:
ids_recibidos = ["C003", "C001", "C003", "C007", "C001", "C002", "C007", "C003"]

unicos = []
for identificador in ids_recibidos:
    if identificador not in unicos:
        unicos.append(identificador)

print("Únicos en orden de llegada:", unicos)
print("Duplicados eliminados:", len(ids_recibidos) - len(unicos))

Únicos en orden de llegada: ['C003', 'C001', 'C007', 'C002']
Duplicados eliminados: 4


## Ejercicio 1.3 · Dividir registros en lotes para una API
El número de lotes se calcula con una división entera "hacia arriba": `(n + t - 1) // t`.

In [3]:
registros = list(range(1, 24))   # 23 registros: 1, 2, ..., 23
tamano_lote = 5

num_lotes = (len(registros) + tamano_lote - 1) // tamano_lote

for n in range(num_lotes):
    lote = registros[n * tamano_lote:(n + 1) * tamano_lote]
    print(f"Lote {n + 1}: {len(lote)} registros -> {lote}")
print("Número de lotes:", num_lotes)

Lote 1: 5 registros -> [1, 2, 3, 4, 5]
Lote 2: 5 registros -> [6, 7, 8, 9, 10]
Lote 3: 5 registros -> [11, 12, 13, 14, 15]
Lote 4: 5 registros -> [16, 17, 18, 19, 20]
Lote 5: 3 registros -> [21, 22, 23]
Número de lotes: 5


## Ejercicio 1.4 · El bug de borrar mientras se recorre
Primero se reproduce el bug del compañero.

In [4]:
importes = [120, -10, -35, 80, -5, -60, 200]
for x in importes:
    if x < 0:
        importes.remove(x)
print(importes)

[120, -35, 80, -60, 200]


**¿Por qué siguen apareciendo -35 y -60?**

El `for` recorre la lista con un índice interno que avanza una posición en cada vuelta. Cuando se elimina un elemento, los siguientes se desplazan una posición a la izquierda, pero el índice sigue avanzando igualmente.

- Con `-10` (índice 1) se elimina el elemento y `-35` pasa a ocupar el índice 1. El `for` continúa en el índice 2, que ahora es `80`, y **`-35` se salta sin evaluarse**.
- Con `-5` ocurre exactamente lo mismo: al borrarlo, `-60` se desplaza y el índice la pasa por alto.

**Corrección:** no modificar la lista que se recorre. Se construye una lista nueva solo con los elementos que se quieren conservar.

In [5]:
importes = [120, -10, -35, 80, -5, -60, 200]
importes = [x for x in importes if x >= 0]
print(importes)

[120, 80, 200]


## Ejercicio 1.5 · Consultas sobre una lista de pedidos

In [6]:
# [pedido_id, tienda, importe, estado]
pedidos = [
    ["P01", "MAD01", 250.0, "ENTREGADO"],
    ["P02", "BCN02", 89.9, "PENDIENTE"],
    ["P03", "MAD01", 1200.0, "ENTREGADO"],
    ["P04", "VLC03", 45.5, "CANCELADO"],
    ["P05", "MAD01", 310.0, "PENDIENTE"],
    ["P06", "BCN02", 760.0, "ENTREGADO"],
    ["P07", "MAD01", 99.0, "ENTREGADO"],
]

# 1. Importes de los pedidos ENTREGADO de MAD01 y su total
importes_mad = [p[2] for p in pedidos if p[1] == "MAD01" and p[3] == "ENTREGADO"]
print("Importes entregados en MAD01:", importes_mad, "-> total", sum(importes_mad))

# 2. Ids de los 3 pedidos de mayor importe
top3 = sorted(pedidos, key=lambda p: p[2], reverse=True)[:3]
print("Top 3 por importe:", [p[0] for p in top3])

# 3. Cambiar PENDIENTE por EN_CURSO en la propia lista
for p in pedidos:
    if p[3] == "PENDIENTE":
        p[3] = "EN_CURSO"
print("Estados actualizados:", [p[3] for p in pedidos])

Importes entregados en MAD01: [250.0, 1200.0, 99.0] -> total 1549.0
Top 3 por importe: ['P03', 'P06', 'P05']
Estados actualizados: ['ENTREGADO', 'EN_CURSO', 'ENTREGADO', 'CANCELADO', 'EN_CURSO', 'ENTREGADO', 'ENTREGADO']


## Ejercicio 1.6 · Reparticionar con round-robin
El registro número `i` va a la partición `i % num_particiones`. El factor de skew es `máximo / media` de los tamaños.

In [7]:
entrada = [
    [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    [113, 114],
    [115, 116, 117, 118, 119, 120],
]
num_particiones = 4

# 1. Juntar todos los registros en una sola lista, en orden
todos = []
for particion in entrada:
    todos.extend(particion)

# 2. Reparto round-robin
salida = [[] for _ in range(num_particiones)]
for i, registro in enumerate(todos):
    salida[i % num_particiones].append(registro)

for i, particion in enumerate(salida):
    print(f"Partición {i}:", particion)

# 3. Skew antes y después
def skew(tamanos):
    return round(max(tamanos) / (sum(tamanos) / len(tamanos)), 2)

antes = [len(p) for p in entrada]
despues = [len(p) for p in salida]
print("Tamaños antes:", antes, "-> skew", skew(antes))
print("Tamaños después:", despues, "-> skew", skew(despues))

Partición 0: [101, 105, 109, 113, 117]
Partición 1: [102, 106, 110, 114, 118]
Partición 2: [103, 107, 111, 115, 119]
Partición 3: [104, 108, 112, 116, 120]
Tamaños antes: [12, 2, 6] -> skew 1.8
Tamaños después: [5, 5, 5, 5] -> skew 1.0


## Ejercicio 1.7 · Ventanas deslizantes sobre un stream de eventos

In [8]:
eventos_por_minuto = [120, 135, 128, 410, 140, 132, 125, 390, 395, 130]
ventana = 3

# 1. Media de cada ventana de 3 minutos consecutivos
medias = [round(sum(eventos_por_minuto[i:i + ventana]) / ventana, 1)
          for i in range(len(eventos_por_minuto) - ventana + 1)]
print("Medias móviles:", medias)
print("Ventanas calculadas:", len(medias))

# 2. Ventana con la media más alta
inicio = medias.index(max(medias))
print(f"Ventana más cargada: minutos {inicio}-{inicio + ventana - 1} (media {max(medias)})")

# 3. Picos: valor > doble de la media de los 3 minutos anteriores (desde el minuto 3)
picos = []
for m in range(ventana, len(eventos_por_minuto)):
    media_previa = sum(eventos_por_minuto[m - ventana:m]) / ventana
    if eventos_por_minuto[m] > 2 * media_previa:
        picos.append(m)
print("Minutos con pico:", picos)

Medias móviles: [127.7, 224.3, 226.0, 227.3, 132.3, 215.7, 303.3, 305.0]
Ventanas calculadas: 8
Ventana más cargada: minutos 7-9 (media 305.0)
Minutos con pico: [3, 7]


---
# Sección 2. Tuplas

## Ejercicio 2.1 · Resumen de pedidos

In [9]:
# (pedido_id, cliente, importe, estado)
pedidos = [
    (5001, "Acme", 250.0, "ENTREGADO"),
    (5002, "Globex", 1340.0, "ENTREGADO"),
    (5003, "Initech", 89.9, "CANCELADO"),
    (5004, "Acme", 610.0, "ENTREGADO"),
    (5005, "Umbrella", 45.0, "CANCELADO"),
    (5006, "Globex", 300.0, "PENDIENTE"),
]

facturacion = 0
cancelados = []
estados = []
for id_pedido, cliente, importe, estado in pedidos:
    estados.append(estado)
    if estado == "ENTREGADO":
        facturacion += importe
    elif estado == "CANCELADO":
        cancelados.append((cliente, importe))

mayor = max(pedidos, key=lambda p: p[2])

print("Facturación entregada:", facturacion)
print(f"Pedido de mayor importe: {mayor[0]} de {mayor[1]}")
print("Cancelados:", cancelados)
print(f"ENTREGADO: {estados.count('ENTREGADO')} | CANCELADO: {estados.count('CANCELADO')} | PENDIENTE: {estados.count('PENDIENTE')}")

Facturación entregada: 2200.0
Pedido de mayor importe: 5002 de Globex
Cancelados: [('Initech', 89.9), ('Umbrella', 45.0)]
ENTREGADO: 3 | CANCELADO: 2 | PENDIENTE: 1


## Ejercicio 2.2 · Validar filas contra un esquema

In [10]:
esquema = (("id", int), ("cliente", str), ("importe", float), ("activo", bool))

filas = [
    (1, "Acme", 120.5, True),
    (2, "Globex", "340", True),
    ("3", "Initech", 99.9, False),
    (4, "Umbrella", 15.0, "no"),
]

errores = 0
for num, fila in enumerate(filas, start=1):
    for (columna, tipo), valor in zip(esquema, fila):
        if not isinstance(valor, tipo):
            print(f"Fila {num}: '{columna}' esperaba {tipo.__name__} y recibió {type(valor).__name__}")
            errores += 1
print(f"Filas: {len(filas)} | Errores: {errores}")

Fila 2: 'importe' esperaba float y recibió str
Fila 3: 'id' esperaba int y recibió str
Fila 4: 'activo' esperaba bool y recibió str
Filas: 4 | Errores: 3


## Ejercicio 2.3 · Comparar versiones de un esquema
Como texto, `"1.10.0"` es menor que `"1.9.7"` porque se compara carácter a carácter (`"1"` < `"9"`). Como tuplas de enteros, `(1, 10, 0) > (1, 9, 7)`.

In [11]:
versiones = ["1.4.2", "1.10.0", "1.9.7", "2.0.0", "1.10.3"]

# 1. Resultado incorrecto como texto
print("Como texto, ¿1.10.0 > 1.9.7?", "1.10.0" > "1.9.7")
print("Ordenadas como texto:", sorted(versiones))

# 2. Cada versión como tupla de enteros
tuplas = [tuple(int(parte) for parte in v.split(".")) for v in versiones]
print("Como tuplas:", tuplas)

# 3. Ordenar, obtener la última y darle formato
ordenadas = sorted(tuplas)
print("Ordenadas:", ordenadas)
print("Última versión: v" + ".".join(str(n) for n in ordenadas[-1]))
print("¿1.10.0 es posterior a 1.9.7?", (1, 10, 0) > (1, 9, 7))

Como texto, ¿1.10.0 > 1.9.7? False
Ordenadas como texto: ['1.10.0', '1.10.3', '1.4.2', '1.9.7', '2.0.0']
Como tuplas: [(1, 4, 2), (1, 10, 0), (1, 9, 7), (2, 0, 0), (1, 10, 3)]
Ordenadas: [(1, 4, 2), (1, 9, 7), (1, 10, 0), (1, 10, 3), (2, 0, 0)]
Última versión: v2.0.0
¿1.10.0 es posterior a 1.9.7? True


## Ejercicio 2.4 · Detectar solapes entre ejecuciones de jobs

In [12]:
from operator import itemgetter

# (job, minuto_inicio, minuto_fin) dentro del día
ejecuciones = [
    ("job_c", 130, 170),
    ("job_a", 60, 95),
    ("job_d", 165, 200),
    ("job_b", 90, 120),
]

orden = sorted(ejecuciones, key=itemgetter(1))
print("Orden de arranque:", [job for job, _, _ in orden])

solapes = 0
for (job_a, ini_a, fin_a), (job_b, ini_b, fin_b) in zip(orden, orden[1:]):
    if ini_b < fin_a:
        print(f"SOLAPE: {job_b} arranca en {ini_b} y {job_a} termina en {fin_a} ({fin_a - ini_b} min)")
        solapes += 1
print("Solapes detectados:", solapes)

Orden de arranque: ['job_a', 'job_b', 'job_c', 'job_d']
SOLAPE: job_b arranca en 90 y job_a termina en 95 (5 min)
SOLAPE: job_d arranca en 165 y job_c termina en 170 (5 min)
Solapes detectados: 2


## Ejercicio 2.5 · MapReduce sobre logs de acceso

In [13]:
from operator import itemgetter

# método ruta código_http bytes
logs = [
    "GET /pedidos 200 5120",
    "GET /clientes 404 310",
    "POST /pedidos 201 880",
    "GET /pedidos 200 4980",
    "GET /stock 500 120",
    "GET /clientes 200 2210",
    "GET /stock 500 95",
    "POST /pedidos 201 910",
]

# MAP: una tupla (código, bytes) por línea, con los bytes como int
pares = []
for linea in logs:
    metodo, ruta, codigo, nbytes = linea.split()
    pares.append((codigo, int(nbytes)))
print("Fase map:", pares[:3], "...")

# SHUFFLE / SORT: ordenar por código
pares.sort(key=itemgetter(0))

# REDUCE: (código, peticiones, bytes_totales) por grupo de claves consecutivas
resultado = []
codigo_actual, peticiones, total = pares[0][0], 0, 0
for codigo, nbytes in pares:
    if codigo == codigo_actual:
        peticiones += 1
        total += nbytes
    else:
        resultado.append((codigo_actual, peticiones, total))
        codigo_actual, peticiones, total = codigo, 1, nbytes
resultado.append((codigo_actual, peticiones, total))

for codigo, peticiones, total in resultado:
    print(f"{codigo}: {peticiones} peticiones, {total} bytes")

Fase map: [('200', 5120), ('404', 310), ('201', 880)] ...
200: 3 peticiones, 12310 bytes
201: 2 peticiones, 1790 bytes
404: 1 peticiones, 310 bytes
500: 2 peticiones, 215 bytes


## Ejercicio 2.6 · Partition pruning con tuplas de fecha
Las tuplas `(año, mes, día)` se comparan elemento a elemento, así que `desde <= fecha <= hasta` funciona directamente.

In [14]:
# (año, mes, día, número de ficheros) de cada partición de la tabla ventas
particiones = [
    (2026, 9, 27, 14), (2026, 9, 28, 12), (2026, 9, 29, 15), (2026, 9, 30, 13),
    (2026, 10, 1, 16), (2026, 10, 2, 11), (2026, 10, 3, 12),
]
desde = (2026, 9, 29)
hasta = (2026, 10, 1)

seleccionadas = [p for p in particiones if desde <= p[:3] <= hasta]

for anio, mes, dia, n_ficheros in seleccionadas:
    print(f"ventas/year={anio}/month={mes:02d}/day={dia:02d} -> {n_ficheros} ficheros")

leidos = sum(p[3] for p in seleccionadas)
total = sum(p[3] for p in particiones)
print(f"Ficheros leídos: {leidos} de {total}")
print(f"Ficheros descartados: {(total - leidos) / total * 100:.1f} %")

ventas/year=2026/month=09/day=29 -> 15 ficheros
ventas/year=2026/month=09/day=30 -> 13 ficheros
ventas/year=2026/month=10/day=01 -> 16 ficheros
Ficheros leídos: 44 de 93
Ficheros descartados: 52.7 %


---
# Sección 3. Set

## Ejercicio 3.1 · Usuarios activos y retención

In [15]:
lunes = {"u01", "u02", "u03", "u04", "u05", "u06"}
martes = {"u02", "u03", "u05", "u07", "u08"}
miercoles = {"u01", "u03", "u05", "u08", "u09"}

print("Activos los tres días:", sorted(lunes & martes & miercoles))
print("Activos al menos un día:", len(lunes | martes | miercoles))
print("Solo el lunes:", sorted(lunes - martes - miercoles))
print("Nuevos el miércoles:", sorted(miercoles - lunes - martes))
retencion = len(lunes & martes) / len(lunes) * 100
print(f"Retención lunes -> martes: {retencion:.1f} %")

Activos los tres días: ['u03', 'u05']
Activos al menos un día: 9
Solo el lunes: ['u04', 'u06']
Nuevos el miércoles: ['u09']
Retención lunes -> martes: 50.0 %


## Ejercicio 3.2 · Detectar datos personales antes de publicar

In [16]:
pii = {"dni", "email", "telefono", "iban", "fecha_nacimiento"}

datasets = [
    ("ventas", {"id_venta", "fecha", "importe", "id_cliente"}),
    ("clientes", {"id_cliente", "nombre", "email", "telefono", "ciudad"}),
    ("pagos", {"id_pago", "iban", "importe", "fecha"}),
    ("logs_web", {"timestamp", "url", "user_agent"}),
]

con_pii = 0
for nombre, columnas in datasets:
    if columnas.isdisjoint(pii):
        print(f"{nombre:<10}sin PII")
    else:
        print(f"{nombre:<10}PII -> {sorted(columnas & pii)}")
        con_pii += 1
print("Datasets que requieren enmascarado:", con_pii)

ventas    sin PII
clientes  PII -> ['email', 'telefono']
pagos     PII -> ['iban']
logs_web  sin PII
Datasets que requieren enmascarado: 2


## Ejercicio 3.3 · ¿Qué transacciones vienen repetidas?

In [17]:
transacciones = ["T10", "T11", "T12", "T11", "T13", "T14", "T12", "T11", "T15"]

vistos = set()
repetidos = set()
for t in transacciones:
    if t in vistos:
        repetidos.add(t)
    else:
        vistos.add(t)

print("Repetidos:", sorted(repetidos))
print(f"Únicos: {len(vistos)} de {len(transacciones)}")

Repetidos: ['T11', 'T12']
Únicos: 6 de 9


## Ejercicio 3.4 · Rutas sin vuelta en un log de vuelos

In [18]:
rutas_hoy = [("MAD", "BCN"), ("BCN", "MAD"), ("MAD", "LIS"), ("MAD", "BCN"),
             ("SVQ", "MAD"), ("LIS", "MAD"), ("BIO", "PMI")]
rutas_manana = [("PMI", "BIO"), ("MAD", "VLC")]

def sin_vuelta(rutas):
    return sorted(r for r in rutas if (r[1], r[0]) not in rutas)

rutas = set(rutas_hoy)
print("Rutas distintas hoy:", len(rutas))
print("Sin vuelta:", sin_vuelta(rutas))

rutas |= set(rutas_manana)
print("Rutas acumuladas:", len(rutas))
print("Sin vuelta tras añadir mañana:", sin_vuelta(rutas))

Rutas distintas hoy: 6
Sin vuelta: [('BIO', 'PMI'), ('SVQ', 'MAD')]
Rutas acumuladas: 8
Sin vuelta tras añadir mañana: [('MAD', 'VLC'), ('SVQ', 'MAD')]


## Ejercicio 3.5 · Particiones que hay que reprocesar

In [19]:
esperadas = {"2026-10-01", "2026-10-02", "2026-10-03", "2026-10-04",
             "2026-10-05", "2026-10-06", "2026-10-07"}
# Carpetas que existen en HDFS
presentes = {"2026-09-30", "2026-10-01", "2026-10-02", "2026-10-03",
             "2026-10-05", "2026-10-06", "2026-10-07"}
# Carpetas que contienen el marcador _SUCCESS (escritura terminada)
con_success = {"2026-09-30", "2026-10-01", "2026-10-02", "2026-10-05", "2026-10-07"}

faltan = esperadas - presentes
incompletas = (esperadas & presentes) - con_success
a_reprocesar = faltan | incompletas
fuera_de_rango = presentes - esperadas
completas = esperadas & con_success

print("Faltan:", sorted(faltan))
print("Incompletas:", sorted(incompletas))
print("A reprocesar:", sorted(a_reprocesar))
print("Fuera de rango:", sorted(fuera_de_rango))
print(f"Completas: {len(completas)} de {len(esperadas)} ({len(completas) / len(esperadas) * 100:.1f} %)")

Faltan: ['2026-10-04']
Incompletas: ['2026-10-03', '2026-10-06']
A reprocesar: ['2026-10-03', '2026-10-04', '2026-10-06']
Fuera de rango: ['2026-09-30']
Completas: 4 de 7 (57.1 %)


## Ejercicio 3.6 · Réplicas vivas de los bloques de HDFS

In [20]:
nodos_vivos = {"dn1", "dn2", "dn4", "dn5"}

# (bloque, set de nodos que guardan una réplica)
bloques = [
    ("blk_001", {"dn1", "dn2", "dn3"}),
    ("blk_002", {"dn3", "dn6", "dn4"}),
    ("blk_003", {"dn2", "dn4", "dn5"}),
    ("blk_004", {"dn3", "dn6", "dn1"}),
    ("blk_005", {"dn6", "dn3", "dn2"}),
    ("blk_006", {"dn3", "dn6"}),
]

todos_los_nodos = set()
en_riesgo = []
perdidos = []
for bloque, nodos in bloques:
    vivas = nodos & nodos_vivos
    print(f"{bloque}: {len(vivas)} réplicas vivas {sorted(vivas)}")
    todos_los_nodos |= nodos
    if len(vivas) == 1:
        en_riesgo.append(bloque)
    elif len(vivas) == 0:
        perdidos.append(bloque)

print("Nodos caídos:", sorted(todos_los_nodos - nodos_vivos))
print("Bloques en riesgo:", en_riesgo)
print("Bloques perdidos:", perdidos)

blk_001: 2 réplicas vivas ['dn1', 'dn2']
blk_002: 1 réplicas vivas ['dn4']
blk_003: 3 réplicas vivas ['dn2', 'dn4', 'dn5']
blk_004: 1 réplicas vivas ['dn1']
blk_005: 1 réplicas vivas ['dn2']
blk_006: 0 réplicas vivas []
Nodos caídos: ['dn3', 'dn6']
Bloques en riesgo: ['blk_002', 'blk_004', 'blk_005']
Bloques perdidos: ['blk_006']


---
# Sección 4. Diccionarios

## Ejercicio 4.1 · Invertir un mapeo

In [21]:
tienda_region = {
    "MAD01": "Centro", "MAD02": "Centro", "BCN01": "Noreste",
    "VLC01": "Levante", "BCN02": "Noreste", "TOL01": "Centro",
}

region_tiendas = {}
for tienda, region in tienda_region.items():
    region_tiendas.setdefault(region, []).append(tienda)

for region in sorted(region_tiendas):
    print(region, "->", sorted(region_tiendas[region]))

Centro -> ['MAD01', 'MAD02', 'TOL01']
Levante -> ['VLC01']
Noreste -> ['BCN01', 'BCN02']


## Ejercicio 4.2 · Total por tienda y mes con claves compuestas

In [22]:
ventas = [
    ("2026-09-15", "MAD01", 200.0),
    ("2026-09-28", "BCN02", 150.0),
    ("2026-10-02", "MAD01", 320.0),
    ("2026-09-30", "MAD01", 80.0),
    ("2026-10-05", "BCN02", 410.0),
    ("2026-10-09", "MAD01", 95.5),
]

totales = {}
for fecha, tienda, importe in ventas:
    clave = (tienda, fecha[:7])
    totales[clave] = totales.get(clave, 0) + importe

for tienda, mes in sorted(totales):
    print(f"{tienda} {mes}: {totales[(tienda, mes)]:.2f}")

BCN02 2026-09: 150.00
BCN02 2026-10: 410.00
MAD01 2026-09: 280.00
MAD01 2026-10: 415.50


## Ejercicio 4.3 · Informe de nulos por campo
Un valor cuenta como nulo si es `None`, una cadena vacía o si la clave no existe (`get()` devuelve `None` en ese caso).

In [23]:
registros = [
    {"id": 1, "email": "ana@acme.com", "ciudad": "Madrid", "edad": 34},
    {"id": 2, "email": None, "ciudad": "Sevilla", "edad": None},
    {"id": 3, "email": "luis@globex.com", "edad": 41},
    {"id": 4, "email": "", "ciudad": None, "edad": 29},
]
campos = ["id", "email", "ciudad", "edad"]

nulos = {campo: 0 for campo in campos}
for registro in registros:
    for campo in campos:
        valor = registro.get(campo)
        if valor is None or valor == "":
            nulos[campo] += 1

for campo in campos:
    porcentaje = nulos[campo] / len(registros) * 100
    print(f"{campo:<8}{nulos[campo]} nulos ({porcentaje:.0f} %)")

id      0 nulos (0 %)
email   2 nulos (50 %)
ciudad  2 nulos (50 %)
edad    1 nulos (25 %)


## Ejercicio 4.4 · Ranking de productos más vendidos

In [24]:
from operator import itemgetter

lineas = [
    ("teclado", 3), ("raton", 10), ("monitor", 2), ("raton", 5),
    ("cable_hdmi", 8), ("teclado", 4), ("monitor", 1), ("cable_hdmi", 9),
]

unidades = {}
for producto, uds in lineas:
    unidades[producto] = unidades.get(producto, 0) + uds
print(unidades)

ranking = sorted(unidades.items(), key=itemgetter(1), reverse=True)
for puesto, (producto, uds) in enumerate(ranking[:3], start=1):
    print(f"{puesto}. {producto} ({uds} uds)")

{'teclado': 7, 'raton': 15, 'monitor': 3, 'cable_hdmi': 17}
1. cable_hdmi (17 uds)
2. raton (15 uds)
3. teclado (7 uds)


## Ejercicio 4.5 · Configuración en tres niveles
En `{**a, **b, **c}` gana el diccionario más a la derecha, así que el orden es defaults, entorno, ejecución.

In [25]:
defaults = {"formato": "csv", "separador": ",", "reintentos": 3, "timeout_s": 60}
entorno = {"formato": "parquet", "timeout_s": 120}
ejecucion = {"reintentos": 5}

config = {**defaults, **entorno, **ejecucion}
print(config)

for clave, valor in config.items():
    if clave in ejecucion:
        nivel = "ejecucion"
    elif clave in entorno:
        nivel = "entorno"
    else:
        nivel = "defaults"
    print(f"{clave:<10} = {str(valor):<8} (viene de {nivel})")

{'formato': 'parquet', 'separador': ',', 'reintentos': 5, 'timeout_s': 120}
formato    = parquet  (viene de entorno)
separador  = ,        (viene de defaults)
reintentos = 5        (viene de ejecucion)
timeout_s  = 120      (viene de entorno)


## Ejercicio 4.6 · Lag de varios grupos de consumidores de Kafka

In [26]:
# Último offset escrito en cada partición del topic "pedidos"
ultimo_offset = {0: 15230, 1: 14980, 2: 15410, 3: 9800}

# Último offset confirmado por cada grupo de consumidores
grupos = {
    "etl_lakehouse": {0: 15230, 1: 14975, 2: 15400, 3: 9800},
    "alertas_fraude": {0: 13900, 1: 14980, 2: 12010, 3: 9790},
    "dashboard_rt": {0: 15229, 1: 14980, 2: 15410},
}
umbral_alerta = 1000

for grupo, confirmados in grupos.items():
    # Si falta la partición, se considera que no ha leído nada (offset 0)
    lags = {p: ultimo_offset[p] - confirmados.get(p, 0) for p in ultimo_offset}
    lag_total = sum(lags.values())
    peor = max(lags, key=lags.get)
    estado = "ALERTA" if lag_total > umbral_alerta else "OK"
    print(f"{grupo:<15} lag {lag_total:>5} | peor partición {peor} ({lags[peor]}) | {estado}")

etl_lakehouse   lag    15 | peor partición 2 (10) | OK
alertas_fraude  lag  4740 | peor partición 2 (3400) | ALERTA
dashboard_rt    lag  9801 | peor partición 3 (9800) | ALERTA


## Ejercicio 4.7 · Simular un broadcast join

In [27]:
# Tabla pequeña: se envía completa a todos los nodos (broadcast)
paises = {"ES": "Europa", "PT": "Europa", "MX": "América", "AR": "América", "JP": "Asia"}

# Tabla grande, ya repartida en 3 particiones: (id_venta, país, importe)
particiones = [
    [("V1", "ES", 120.0), ("V2", "MX", 80.0), ("V3", "ES", 45.5)],
    [("V4", "JP", 300.0), ("V5", "AR", 60.0), ("V6", "FR", 99.0)],
    [("V7", "PT", 210.0), ("V8", "MX", 150.0)],
]

globales = {}
for i, particion in enumerate(particiones):
    # Cada partición se cruza localmente con el diccionario broadcast
    local = {}
    for id_venta, pais, importe in particion:
        continente = paises.get(pais, "Desconocido")
        local[continente] = local.get(continente, 0) + importe
    print(f"Partición {i}:", local)

    # Fusión del resultado local en el total global
    for continente, total in local.items():
        globales[continente] = globales.get(continente, 0) + total

print("Ventas por continente:")
for continente in sorted(globales):
    print(f"  {continente:<12}{globales[continente]:>8.2f}")

Partición 0: {'Europa': 165.5, 'América': 80.0}
Partición 1: {'Asia': 300.0, 'América': 60.0, 'Desconocido': 99.0}
Partición 2: {'Europa': 210.0, 'América': 150.0}
Ventas por continente:
  América       290.00
  Asia          300.00
  Desconocido    99.00
  Europa        375.50
